# netMHCpan
![netMHCpan](https://proto-bio.github.io/proto-assets/images/tool/netmhcpan/hero.png)

This notebook demonstrates the two netMHCpan tools for MHC class I epitope prediction.
`run_netmhcpan_peptides` scores peptides you already have against a panel of MHC class I
alleles, and `run_netmhcpan_protein` windows a protein into class I k-mers and scores every
window, which is the usual way to screen a designed or natural sequence for epitopes.
netMHCpan is an ensemble of neural networks trained jointly on binding-affinity (BA) and
mass-spectrometry eluted-ligand (EL) data, so a single pan-specific model covers any allele
with a known sequence. It uses structural input features, offers two transfer-learned heads
for pathogen epitopes and neoepitopes (`include_pathogen`, `include_neoepitope`), and can
encode flanking context on the protein scan (`use_context`). Predictions come back as a %rank against a background of random
peptides, which is what makes them comparable across alleles.

Learn more at [services.healthtech.dtu.dk/services/NetMHCpan-4.2](https://services.healthtech.dtu.dk/services/NetMHCpan-4.2/)
and in the method paper: Nilsson, Greenbaum, Peters & Nielsen, *NetMHCpan-4.2: improved
prediction of CD8+ epitopes by use of transfer learning and structural features*,
Frontiers in Immunology 16:1616113 (2025),
[doi:10.3389/fimmu.2025.1616113](https://doi.org/10.3389/fimmu.2025.1616113).

**Note:** netMHCpan is license-gated and user-provisioned. It is not bundled with proto-tools
and cannot be downloaded automatically: the DTU academic license requires you to register and
accept the terms yourself, and forbids redistributing the package. To run the cells below you
must install netMHCpan 4.2 locally and either set the `NETMHCPAN_HOME` environment variable to
the install root or pass `netmhcpan_dir` in the config. See the toolkit's `SETUP.md` for the
provisioning steps. Every run cell in this notebook errors without a local install.

In [ ]:
from proto_tools.tools import (
    NetMHCpanPeptidesConfig,
    NetMHCpanPeptidesInput,
    NetMHCpanProteinConfig,
    NetMHCpanProteinInput,
    run_netmhcpan_peptides,
    run_netmhcpan_protein,
)
from proto_tools.utils.notebook_docs import display_api_reference

In [ ]:
# Display input API reference
display_api_reference("netmhcpan-peptides", "input", "run_netmhcpan_peptides")

# Four well-characterised class I peptides:
#   AAAWYLWEV - HIV-1 reverse transcriptase, a known HLA-A*02:01 binder
#   GILGFVFTL - influenza A M1(58-66), the canonical HLA-A*02:01 epitope
#   KLGGALQAK - human CMV IE1(316-324), HLA-A*03:01-restricted (an A*02:01 non-binder)
#   AAGLQDCTM - no reported class I restriction; included as a negative control
inputs = NetMHCpanPeptidesInput(
    peptides=["AAAWYLWEV", "GILGFVFTL", "KLGGALQAK", "AAGLQDCTM"],
)

In [ ]:
# Display config API reference
display_api_reference("netmhcpan-peptides", "config", "run_netmhcpan_peptides")

# A three-allele panel. netMHCpan itself accepts only the unstarred spelling, but the wrapper
# normalizes the starred forms ("HLA-A*02:01", "A*02:01") and bare human loci ("A02:01") too,
# so all the spellings in common use converge on the same allele.
config = NetMHCpanPeptidesConfig(
    alleles=["HLA-A*02:01", "HLA-A01:01", "HLA-B07:02"],
)
print(f"Normalized allele panel: {config.alleles}")

In [ ]:
# Display output API reference
display_api_reference("netmhcpan-peptides", "output", "run_netmhcpan_peptides")

In [ ]:
# Score every peptide against every allele in the panel (one netMHCpan invocation).
peptide_result = run_netmhcpan_peptides(inputs, config)
print(f"netMHCpan version: {peptide_result.netmhcpan_version}")

# One PeptidePrediction per input peptide, index-aligned with inputs.peptides. Each carries
# one binding record per configured allele, plus that peptide's summary metrics.
for prediction in peptide_result.predictions:
    print(f"\n{prediction.peptide}")
    for binding in prediction.bindings:
        print(
            f"  {binding.allele:<12} rank_el={binding.rank_el:>7.3f}  "
            f"score_el={binding.score_el:.6f}  {binding.binder_class}"
        )
    print(
        f"  summary: strong={prediction.strong_binder_count} weak={prediction.weak_binder_count} "
        f"best_rank={prediction.best_rank:.3f} binding_propensity={prediction.binding_propensity:.4f}"
    )

In [ ]:
# Human beta-2 microglobulin (mature chain, 99 residues) - the class I light chain itself.
B2M = (
    "IQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDW"
    "SFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM"
)

# The protein tool does the windowing for you: every k-mer of every configured length is
# enumerated and scored. Here 9- and 10-mers against two alleles.
protein_inputs = NetMHCpanProteinInput(sequences=[B2M])
protein_config = NetMHCpanProteinConfig(
    alleles=["HLA-A02:01", "HLA-B07:02"],
    peptide_lengths=[9, 10],
)
protein_result = run_netmhcpan_protein(protein_inputs, protein_config)

scan = protein_result.results[0]
print(f"Sequence length: {scan.sequence_length} residues")
print(f"Windows scored:  {scan.num_windows}")
print(f"Binding records: {len(scan.bindings)}  (one per window per allele)")
print(
    f"Strong binders: {scan.strong_binder_count}, weak: {scan.weak_binder_count}, "
    f"best rank: {scan.best_rank:.3f}"
)
print(f"Strong binders per 100 residues per allele: {scan.strong_binders_per_100_residues:.2f}")

# The strongest predicted epitopes. `position` is 1-indexed against the supplied sequence.
print("\nTop 8 windows by EL %rank:")
for binding in sorted(scan.bindings, key=lambda b: b.rank_el)[:8]:
    print(
        f"  pos {binding.position:>3}  {binding.peptide:<11} {binding.allele:<12} "
        f"rank_el={binding.rank_el:>6.3f}  core={binding.core:<10} {binding.binder_class}"
    )

In [ ]:
# Advanced usage: a wider allele panel, binding affinity alongside the default eluted-ligand
# output, the two 4.2 transfer-learned heads, and stricter binder thresholds.
#
#   include_binding_affinity=True passes netMHCpan's -BA flag, which *adds* score_ba, rank_ba
#   and affinity_nm to the EL columns rather than replacing them.
#   include_pathogen / include_neoepitope are additive in the same way, each adding its own
#   score and %rank from a head fine-tuned on IEDB pathogen epitopes or CEDAR neoepitopes.
#   The authors report the fine-tuning gain is modest and does not transfer between the two,
#   so match the head to the task and keep ranking on rank_el.
#   strong_rank / weak_rank only change how rank_el is classified; they do not change any score.
advanced_config = NetMHCpanPeptidesConfig(
    alleles=["HLA-A*02:01", "HLA-A01:01", "HLA-A03:01", "HLA-B07:02"],
    include_binding_affinity=True,
    include_pathogen=True,
    include_neoepitope=True,
    strong_rank=0.2,
    weak_rank=1.0,
)
advanced_result = run_netmhcpan_peptides(inputs, advanced_config)

for prediction in advanced_result.predictions:
    print(f"\n{prediction.peptide}")
    for binding in prediction.bindings:
        print(
            f"  {binding.allele:<12} rank_el={binding.rank_el:>7.3f}  "
            f"rank_ba={binding.rank_ba:>7.3f}  affinity={binding.affinity_nm:>9.2f} nM  "
            f"rank_pathogen={binding.rank_pathogen:>7.3f}  rank_neo={binding.rank_neo:>7.3f}  "
            f"{binding.binder_class}"
        )

# The BA fields are populated only when -BA was requested; in the default EL-only run above
# they are None. Same peptide, same allele, two runs:
default_binding = peptide_result.predictions[0].bindings[0]
ba_binding = advanced_result.predictions[0].bindings[0]
print(f"peptide={default_binding.peptide} allele={default_binding.allele}")
print(f"  EL-only run: rank_el={default_binding.rank_el}, rank_ba={default_binding.rank_ba}, "
      f"affinity_nm={default_binding.affinity_nm}")
print(f"  -BA run:     rank_el={ba_binding.rank_el}, rank_ba={ba_binding.rank_ba}, "
      f"affinity_nm={ba_binding.affinity_nm}")

# rank_el is identical across the two runs: -BA adds columns, it does not change the EL
# prediction. What did change is binder_class, because the thresholds were tightened.
print(f"\nrank_el unchanged: {default_binding.rank_el == ba_binding.rank_el}")
print(
    f"binder_class at strong_rank=0.5: {default_binding.binder_class}  ->  "
    f"at strong_rank=0.2: {ba_binding.binder_class}"
)

In [ ]:
# 4.2 can encode the residues flanking each window (netMHCpan's -context), which carry a
# processing signal the peptide alone does not. It is available on the protein scan because
# the flanks are read straight off the sequence you supplied; peptide mode would require you
# to provide each peptide's flanks yourself.
#
# Context re-scores every window, so a context run and a plain run are not comparable -
# pick one per experiment rather than mixing their numbers.
context_config = NetMHCpanProteinConfig(
    alleles=["HLA-A02:01", "HLA-B07:02"],
    peptide_lengths=[9, 10],
    use_context=True,
)
context_scan = run_netmhcpan_protein(protein_inputs, context_config).results[0]

print(f"Windows scored: {context_scan.num_windows} (unchanged by context)")
print(
    f"Strong binders without context: {scan.strong_binder_count}, "
    f"with context: {context_scan.strong_binder_count}"
)
print(f"Best rank without context: {scan.best_rank:.3f}, with context: {context_scan.best_rank:.3f}")

# The same windows, re-ranked. Position and peptide are unchanged; the scores are not.
print("\nTop 5 windows with context encoding:")
for binding in sorted(context_scan.bindings, key=lambda b: b.rank_el)[:5]:
    print(
        f"  pos {binding.position:>3}  {binding.peptide:<11} {binding.allele:<12} "
        f"rank_el={binding.rank_el:>6.3f}  {binding.binder_class}"
    )


In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# CSV gives a flat table, one row per peptide-allele (or window-allele) prediction - the form
# you want for downstream filtering and plotting.
advanced_result.export(name="netmhcpan_peptides", export_path=output_dir, file_format="csv")
protein_result.export(name="netmhcpan_protein_scan", export_path=output_dir, file_format="csv")

# JSON preserves the full nested structure: per-item summary metrics, every binding record,
# and the netMHCpan version that produced them.
advanced_result.export(name="netmhcpan_peptides", export_path=output_dir, file_format="json")

for path in sorted(output_dir.glob("netmhcpan_*")):
    print(f"{path.name}  ({path.stat().st_size} bytes)")

print("\nFirst rows of the peptide CSV:")
print("\n".join((output_dir / "netmhcpan_peptides.csv").read_text().splitlines()[:5]))